In [ ]:
import cv2
import time
import mediapipe as mp
from ugot import ugot


# =========================
# UGOT SETUP
# =========================

got = ugot.UGOT()
got.initialize("192.168.88.1")
# =========================
# MEDIAPIPE SETUP
# =========================

BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
RunningMode = mp.tasks.vision.RunningMode

options = HandLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path="hand_landmarker.task"
    ),
    running_mode=RunningMode.VIDEO,
    num_hands=1
)

landmarker = HandLandmarker.create_from_options(options)


# =========================
# HAND CONNECTIONS
# =========================

HAND_CONNECTIONS = [
    (0, 1), (1, 2), (2, 3), (3, 4),          # Thumb
    (0, 5), (5, 6), (6, 7), (7, 8),          # Index
    (5, 9), (9, 10), (10, 11), (11, 12),     # Middle
    (9, 13), (13, 14), (14, 15), (15, 16),   # Ring
    (13, 17), (17, 18), (18, 19), (19, 20),  # Pinky
    (0, 17)
]


# =========================
# GESTURE DETECTION
# =========================

def detect_gesture(hand_landmarks):

    wrist = hand_landmarks[0]
    # Finger tips
    index_tip = hand_landmarks[8]
    middle_tip = hand_landmarks[12]
    ring_tip = hand_landmarks[16]
    # Finger joints
    index_joint = hand_landmarks[6]
    middle_joint = hand_landmarks[10]
    ring_joint = hand_landmarks[14]
    # Check if fingers are up
    INDEX_UP = index_tip.y < index_joint.y
    MIDDLE_UP = middle_tip.y < middle_joint.y
    RING_UP = ring_tip.y < ring_joint.y
    # =========================
    # GESTURES
    # =========================
    
    # if INDEX_UP and MIDDLE_UP and RING_UP:
    #     got.mecanum_turn_speed(3, 30)
    #     return "Three"
    # elif INDEX_UP and MIDDLE_UP:
    #     got.mecanum_turn_speed(2,30)
    #     return "Two"
    # elif INDEX_UP:
    #     got.mecanum_move_speed(0,15)
    #     return "One"
    if wrist.y < index_tip.y:
        got.mecanum_move_xyz(0, -30, 0)
        return "down"

    elif INDEX_UP:
        index_x = index_tip.x
        index_y = index_tip.y
        y_speed = 30 - int(index_y * 30)
        z_speed = 50 - int(index_x * 100)
        if -10 < z_speed < 10: # your finger is in the middle
            got.mecanum_move_xyz(0, y_speed, 0) # don't turn
        else:
            got.mecanum_move_xyz(0, y_speed, z_speed)
        return f"one, {index_x:.2f}, {index_y:.2f}"
    
    else:    
        got.mecanum_stop()
        return "UNKNOWN"


# =========================
# CAMERA
# =========================

camera = cv2.VideoCapture(0)
camera.set(cv2.CAP_PROP_FRAME_WIDTH, 1920)
camera.set(cv2.CAP_PROP_FRAME_HEIGHT, 1080)

start_time = time.time()

while True:

    success, frame = camera.read()

    if not success:
        print("Could not read camera")
        break

    # Mirror the camera
    frame = cv2.flip(frame, 1)

    height, width, channels = frame.shape


    # =========================
    # CONVERT OPENCV -> MEDIAPIPE
    # =========================

    rgb_frame = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_frame
    )


    # =========================
    # TIMESTAMP
    # =========================

    timestamp_ms = int(
        (time.time() - start_time) * 1000
    )


    # =========================
    # DETECT HANDS
    # =========================

    result = landmarker.detect_for_video(
        mp_image,
        timestamp_ms
    )


    # =========================
    # DRAW HANDS
    # =========================

    if result.hand_landmarks:

        for hand_landmarks in result.hand_landmarks:

            # =========================
            # DETECT GESTURE
            # =========================

            gesture = detect_gesture(
                hand_landmarks
            )


            # =========================
            # DRAW CONNECTIONS
            # =========================

            for start, end in HAND_CONNECTIONS:

                x1 = int(
                    hand_landmarks[start].x * width
                )

                y1 = int(
                    hand_landmarks[start].y * height
                )

                x2 = int(
                    hand_landmarks[end].x * width
                )

                y2 = int(
                    hand_landmarks[end].y * height
                )

                cv2.line(
                    frame,
                    (x1, y1),
                    (x2, y2),
                    (255, 255, 255),
                    2
                )


            # =========================
            # DRAW LANDMARK POINTS
            # =========================

            for landmark_id, landmark in enumerate(hand_landmarks):

                x = int(
                    landmark.x * width
                )

                y = int(
                    landmark.y * height
                )

                # Draw landmark dot
                cv2.circle(
                    frame,
                    (x, y),
                    6,
                    (0, 255, 0),
                    -1
                )

                # Draw landmark number
                cv2.putText(
                    frame,
                    str(landmark_id),
                    (x + 5, y - 5),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.4,
                    (0, 0, 255),
                    1
                )


            # =========================
            # HIGHLIGHT INDEX FINGER
            # =========================

            index_tip = hand_landmarks[8]

            index_x = int(
                index_tip.x * width
            )

            index_y = int(
                index_tip.y * height
            )

            cv2.circle(
                frame,
                (index_x, index_y),
                12,
                (0, 255, 255),
                3
            )


            # =========================
            # FIND TOP OF HAND
            # =========================

            top_landmark = min(
                hand_landmarks,
                key=lambda landmark: landmark.y
            )

            text_x = int(
                top_landmark.x * width
            )

            text_y = int(
                top_landmark.y * height
            ) - 25


            # =========================
            # TEXT SIZE
            # =========================

            (text_width, text_height), baseline = cv2.getTextSize(
                gesture,
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                2
            )


            # =========================
            # KEEP TEXT ON SCREEN
            # =========================

            if text_x + text_width > width:
                text_x = width - text_width - 10

            if text_x < 10:
                text_x = 10

            if text_y - text_height < 10:
                text_y = text_height + 10


            # =========================
            # DRAW TEXT BACKGROUND
            # =========================

            cv2.rectangle(
                frame,
                (
                    text_x - 5,
                    text_y - text_height - 5
                ),
                (
                    text_x + text_width + 5,
                    text_y + baseline + 5
                ),
                (0, 0, 0),
                -1
            )


            # =========================
            # SHOW GESTURE ON HAND
            # =========================

            cv2.putText(
                frame,
                gesture,
                (text_x, text_y),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.8,
                (0, 255, 255),
                2
            )


    # =========================
    # INSTRUCTIONS
    # =========================

    cv2.putText(
        frame,
        "Press Q to quit",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255, 255, 255),
        2
    )


    # =========================
    # SHOW CAMERA
    # =========================

    cv2.imshow(
        "UGOT MediaPipe Hand Controller",
        frame
    )


    # =========================
    # QUIT
    # =========================

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


# =========================
# CLEAN UP
# =========================

camera.release()
cv2.destroyAllWindows()
landmarker.close()